# Threshold Cox with Gaussian Noise (Demonstration)

## Introduction

This page is a demonstration. The previous page, the [Cox threshold
page](cox-threshold.qmd), fits the Cox model under threshold FHE: the
fit matches the cleartext fit, no single party can decrypt, and the
aggregator sees only the joint log-likelihood $\ell(\beta)$ at each
optimizer query.

Here we ask what happens if each site also adds Gaussian noise of the
kind an *output differential privacy* mechanism uses. The sensitivity
$\Delta = 1$ below is a placeholder, so the $\varepsilon$ values are not
a privacy guarantee. Adding the noise takes one extra `rng.normal()`
call per site per query. With the optimizers used here, the fits are
poor at any noise level that gives a small $\varepsilon$. The page runs
the fits and reports the numbers.

## A brief output-DP primer

Given a query $f: \mathcal{D} \to \mathbb{R}$ on a dataset $D$, the
*Gaussian mechanism* (Dwork and Roth 2014, sec. 3.5.3) releases
$\tilde f(D) = f(D) + \mathcal{N}(0, \sigma^2)$. If $f$’s *sensitivity*
(the largest change $f$ can undergo when one record is added or removed)
is $\Delta$, then for any $\varepsilon \in (0, 1)$ and $\delta > 0$,
choosing $\sigma \geq \Delta \sqrt{2\ln(1.25/\delta)}/\varepsilon$ makes
a single release of $\tilde f$ satisfy $(\varepsilon, \delta)$
differential privacy.

The optimizer issues many queries, so the per-query budget composes. We
use *zCDP composition* (Bun and Steinke 2016): the Gaussian mechanism is
$\rho$-zCDP with $\rho = (\Delta/\sigma)^2/2$; $T$ queries compose
linearly to $T \cdot \rho$ in zCDP units; and that converts to
$(\varepsilon, \delta)$ via
$\varepsilon = \rho + 2\sqrt{\rho \log(1/\delta)}$.

## The protocol

The setup is identical to the [Cox threshold page](cox-threshold.qmd),
except each site adds an independent Gaussian noise term to its local
contribution before encryption. Since variances of independent normals
add, the decrypted total is $\ell(\beta) + \mathcal{N}(0, \sigma^2)$
when each site adds $\mathcal{N}(0, \sigma^2/N)$.

## The Cox setup (same DLBCL data as the other Cox pages)

In [2]:
import numpy as np
from homomorphepy import load_dlbcl, site_order
from statsmodels.duration.hazard_regression import PHReg

df = load_dlbcl()
COVARIATES = ["GCB_sig", "LN_sig", "Prolif_sig", "BMP6", "MHC2_sig"]

def site_frame(name):
    m = (df["Subgroup"] == name).to_numpy()
    return {
        "time":   df["time"].to_numpy(dtype=float)[m],
        "status": df["status"].to_numpy(dtype=int)[m],
        "X":      np.column_stack([df[c].to_numpy(dtype=float)[m]
                                   for c in COVARIATES]),
    }

cox_data = {name: site_frame(name) for name in site_order()}

agg_model = PHReg(df["time"], df[COVARIATES], status=df["status"],
                  strata=df["Subgroup"].cat.codes, ties="efron")
agg_coef  = np.asarray(agg_model.fit().params, dtype=float)

def local_cox_nll(data, beta):
    b = np.asarray(beta, dtype=float).ravel()
    model = PHReg(data["time"], data["X"],
                  status=data["status"], ties="efron")
    return float(-model.loglike(b))

## Threshold setup and DP-noised workers

The threshold setup is also the same as on the Cox threshold page. The
only change is in each worker’s local function: it adds an independent
$\mathcal{N}(0, \sigma^2/N)$ draw to its local nLL before returning it.
The noisy value is then encrypted as usual.

In [3]:
import inspect
from homomorphepy.examples import dp

print(inspect.getsource(dp.fit_at_sigma))

def fit_at_sigma(
    sigma: float,
    method: str = "Nelder-Mead",
    seed: int = 1,
    delta: float = DEFAULT_DELTA,
) -> DPFit:
    """Fit the threshold-Cox model with per-site Gaussian noise.

    Each site adds ``N(0, sigma^2/N)`` to its local negative
    log-likelihood before encryption, so the decrypted aggregate
    carries ``N(0, sigma^2)``.
    """
    sites = _cox_sites()
    names = list(sites)
    n_sites = len(names)
    rng = np.random.default_rng(seed)
    scale = sigma / math.sqrt(n_sites)

    def noisy_local(data, beta):
        value = local_cox_nll(data, beta)
        if value is None or math.isnan(value):
            return math.nan
        return value + (rng.normal(0.0, scale) if sigma > 0 else 0.0)

    ctx = fhe_context("CKKS", **CKKS_PARAMS)
    workers = [make_worker(n, sites[n], noisy_local) for n in names]
    master = make_threshold_master("Aggregator", ctx, workers)

    calls = {"n": 0}

    def objective(beta):
        calls["n"] += 1
        v = m

Each fit is a full optimizer run through the encrypted channel, so the
sweep is recorded once rather than run on every render:

``` bash
OMP_NUM_THREADS=2 uv run python docs/_recorded/record_dp_sweep.py
```

The recorded sweep took 1.6 minutes.

## Mechanical correctness: $\sigma = 0$ reproduces the threshold fit

When the noise is zero the protocol reduces to the lossless threshold
protocol. The fitted coefficients match the centralized fit to
threshold-CKKS precision.

The maximum absolute coefficient difference at $\sigma = 0$ is
$3.84 \times 10^{-7}$. So any deviation from the centralized fit below
comes from the noise.

## Increasing noise: where does it break?

We sweep $\sigma$ across five orders of magnitude. Each fit is one full
BFGS run over the threshold-DP encrypted nLL. BFGS estimates the
gradient by finite differences of these nLL values.

BFGS does not move from the zero starting point at any $\sigma > 0$; the
rows show the starting values.

## Nelder–Mead at the same noise scales

The same sweep with Nelder–Mead. The starting simplex adds 0.1 to one
coordinate per vertex. The search stops when the spread of objective
values over the simplex is at most $1 \times 10^{-7}$ relative to the
starting value, or after 500 evaluations.

Fidelity decays as $\sigma$ grows.

## Privacy budget

For the fits at the first three values of $\sigma$, with sensitivity
$\Delta = 1$ (placeholder) and target $\delta = 10^{-5}$, zCDP
composition gives:

In [8]:
def zcdp_to_eps(rho, delta=1e-5):
    return rho + 2 * np.sqrt(rho * np.log(1 / delta))

first_three = sorted(s for s in rows["sigma"].unique() if s > 0)[:3]
budget = pd.concat([
    rows[(rows["method"] == m) & rows["sigma"].isin(first_three)][["method", "sigma", "n_queries"]]
    for m in ("BFGS", "Nelder-Mead")])
budget["rho_per_query"] = (1 / budget["sigma"]) ** 2 / 2
budget["rho_total"]     = budget["n_queries"] * budget["rho_per_query"]
budget["epsilon_at_delta_1e_minus_5"] = zcdp_to_eps(budget["rho_total"])

The $\varepsilon$ column shows that at noise levels where the fits are
still close, $\varepsilon$ is large. Whether that is acceptable depends
on the application.

One could also explore tighter sensitivity bound $\Delta$ or allow far
fewer queries, etc. We don’t do that here.

Bun, Mark, and Thomas Steinke. 2016. “Concentrated Differential Privacy:
Simplifications, Extensions, and Lower Bounds.” *Theory of Cryptography
Conference (TCC)*, 635–58.
<https://doi.org/10.1007/978-3-662-53641-4_24>.

Dwork, Cynthia, and Aaron Roth. 2014. *The Algorithmic Foundations of
Differential Privacy*. Foundations and Trends in Theoretical Computer
Science 9(3–4). Now Publishers. <https://doi.org/10.1561/0400000042>.